# Chapter 7 — Distributed Systems
### ML Engineer (Production) Interview Playbook

**Topics:** Redis · Kafka · RabbitMQ · Queues · Retry · DLQ · Event-Driven · Scaling

> Production services run distributed at scale: multiple instances, multiple services, queues and
> message brokers. This matters enormously for an ML Engineer because data and inference pipelines are
> usually async and event-driven. This chapter is the heaviest theory section — mastery of queues,
> delivery guarantees, resilience, and scalability is what separates a junior engineer from a senior
> one.
>
> **Interview tip:** The core mindset of distributed systems: **"everything fails"** — networks drop,
> messages get lost or duplicated, services slow down or go dark. Good design assumes failure and plans
> for it, rather than hoping it won't happen.

## 7.1 Redis

Redis is an in-memory data store; because data lives in RAM, latency is sub-millisecond. Its operations
(mostly single-threaded) are atomic, which makes it ideal for counters and locks. Common uses:

- **Cache** — caching expensive results (the most common use case).
- **Rate limiting** — a shared atomic counter across instances.
- **Session / feature store** — fast feature storage for real-time inference.
- **Pub/Sub and lightweight queueing** — simple messaging and Redis Streams.

In [ ]:
# Cache-aside pattern (lazy loading)
def get_features(uid: str) -> dict:
    cached = redis.get(f"feat:{uid}")
    if cached:
        return json.loads(cached)          # cache hit

    feats = compute_features(uid)          # cache miss -> expensive computation
    redis.setex(f"feat:{uid}", 300, json.dumps(feats))  # TTL = 5 minutes
    return feats


### Persistence and eviction

- **Persistence:** RDB (periodic snapshot) and AOF (a log of every write). RDB is faster and lighter,
  AOF is more durable. The two can also be combined.
- **Eviction:** when memory fills up, policies like `allkeys-lru` (evict least-recently-used) or
  `volatile-ttl` decide what gets removed.

### Cache challenges

| Problem | Description | Fix |
|---|---|---|
| Cache Stampede | A hot key expires and a flood of requests hits the source at once | A rebuild lock, TTL with jitter, proactive refresh |
| Stale Data | Cached data drifts out of sync with the source | Appropriate TTL, invalidation on write |
| Cache Penetration | Repeated requests for a key that doesn't exist | Cache the empty result too |

**Note:** The hardest cache problem is **cache invalidation**. Two main strategies: TTL (simple, but
has a window of stale data) and explicit invalidation on write (more accurate, but more complex and
bug-prone). The choice depends on how much staleness is tolerable.

### Q7.1 — When do you use Redis as a cache, and how do you avoid stale data?

When computing or fetching data is expensive, the same data is read repeatedly, and some staleness is
tolerable (e.g. a user's features for scoring). I typically use the cache-aside pattern: check the
cache first, compute and store with a TTL on a miss. For staleness: I set a TTL proportional to how
often the data changes, and if accuracy matters, I invalidate or update the cache key when the source
is written. For hot keys, I randomize the TTL with jitter to avoid a cache stampede (simultaneous
expiry causing a flood to the source). It's important to remember the cache is an optimization, not the
source of truth — the system must work correctly even without it.

## 7.2 Queues and Messaging

A queue decouples work from a producer and hands it to a consumer. This "decoupling" has three big
benefits: (1) the producer doesn't wait for processing (fast response), (2) against a load spike, the
queue acts as a buffer, (3) if the consumer is temporarily down, the work stays in the queue and isn't
lost.

| Communication | How | Trade-off |
|---|---|---|
| Sync | Direct call, wait for a response | Simple but coupled and fragile against slowness/downtime of the target |
| Async | Message in a queue, processed later | Resilient and scalable, but more complex (eventual consistency) |

In ML architectures, queues are the backbone: batch inference, scheduled retraining, and data pipelines
typically run on queues to stay decoupled and scalable.

## 7.3 Kafka vs. RabbitMQ

Both are message brokers but have fundamentally different philosophies; knowing the difference is the
classic question in this domain.

| Feature | Kafka | RabbitMQ |
|---|---|---|
| Model | Distributed, durable log | Message broker (AMQP) |
| After consumption | Message remains (until retention expires) | Message deleted after ack |
| Replay | Yes, by rewinding the offset | No (not simply) |
| Throughput | Very high | High |
| Routing | Simple (topic/partition-based) | Powerful (exchange and routing key) |
| Best for | Event streaming, data pipelines | Task queues, async RPC |

### Key Kafka concepts

- **Topic and Partition:** each topic is split into several partitions; partitioning is the foundation
  of parallelism and scale.
- **Consumer Group:** several consumers in one group split the work across partitions (horizontal
  scaling of consumption).
- **Offset:** each consumer's read position; because messages persist, you can rewind the offset and
  replay.

**Interview tip:** Rule of thumb: if several independent consumers need to read the same stream, you
need replay, or throughput is very high → Kafka. If you want work distribution (a work queue) with
flexible routing and transient messages → RabbitMQ. Kafka for a long-lived "event log," RabbitMQ for a
"work queue."

### Q7.3 — Explain the difference between Kafka and RabbitMQ, and give an example of each.

Kafka is a durable, distributed log: messages remain after consumption (until retention expires),
several independent consumer groups can read the same stream, and you can replay by rewinding the
offset; it has very high throughput. Example: a stream of transaction events that the fraud service,
the data warehouse, and the model retraining system all independently read from. RabbitMQ is a
traditional AMQP-based broker: a message is deleted after ack, it has powerful routing via exchanges
and routing keys, and it's excellent for distributing work among workers. Example: a notification-
sending queue consumed by a pool of workers. Core difference: Kafka is a "replayable event log,"
RabbitMQ is a "transient work queue." 

## 7.4 Delivery Guarantees and Message Ordering

One of the most important concepts in message-driven systems: how many times is each message
delivered?

| Guarantee | Meaning | Consequence |
|---|---|---|
| At-most-once | Zero or one time | A message can be lost; never duplicated |
| At-least-once | One or more times | No message is lost, but duplicates are possible → the consumer must be idempotent |
| Exactly-once | Exactly one time | Ideal, but expensive and complex; limited in practice |

**Note:** In practice, at-least-once is the most common because it balances reliability and complexity
well. Direct consequence: the consumer must be idempotent so duplicate processing doesn't cause a
problem (e.g. by recording the ids of already-processed messages). "Exactly-once" in reality often
means "at-least-once + idempotent processing."

### Message ordering

Global ordering is expensive in a distributed system. Kafka only guarantees ordering **within a single
partition**, not across partitions. So if ordering matters for a given entity (e.g. events for one
account), all messages for that entity must go to the same partition using one key.

### Q7.4 — What does at-least-once mean, and what responsibility does it place on the consumer?

It means each message is delivered at least once (and sometimes more than once, e.g. when a consumer
processes a message but crashes before recording the ack); so no message is lost, but a duplicate might
arrive. The responsibility falls on the consumer to be idempotent: reprocessing a message must not
cause a doubled effect. Common approaches: keeping track of processed message ids and rejecting
duplicates, or designing the operation to be inherently idempotent (e.g. upsert instead of insert, or
setting an absolute value instead of a relative increment). This connects directly to the idempotency
discussion in the API Design chapter.

## 7.5 Retry and Dead Letter Queue

Errors are unavoidable in a distributed system. The first step is distinguishing the error type:

- **Transient error:** temporary and fixable by retrying (a network timeout, a temporarily busy
  service). → Retry makes sense.
- **Permanent error:** not fixed by retrying (a malformed message, invalid data). → Retry is useless
  and harmful.

In [ ]:
import time, random

def with_retry(fn, retries=3, base=0.5):
    for attempt in range(retries):
        try:
            return fn()
        except PermanentError:
            raise                          # don't retry
        except TransientError:
            if attempt == retries - 1:
                raise
            # exponential backoff + jitter to avoid a thundering herd
            sleep = base * (2 ** attempt) + random.uniform(0, 0.1)
            time.sleep(sleep)


**Interview tip:** Why jitter? If a thousand consumers fail at the same moment and all retry with
the exact same backoff, they all hit the recovering service in one wave at once (a thundering herd) and
knock it back down. Jitter (small randomization) spreads that wave out.

### Dead Letter Queue (DLQ)

Messages that keep failing after the retry limit are moved to a separate queue (the DLQ). This solves
two problems: (1) the main queue doesn't get blocked and the rest of the messages keep processing,
(2) problematic messages aren't lost and can be inspected, fixed, and reprocessed later.

**Note:** A "poison message" is one that always fails. Without a DLQ, this message (under
at-least-once delivery) gets retried infinitely and blocks the queue forever. A retry cap + DLQ is the
standard solution. The DLQ needs monitoring and alerting, or silent problems will pile up in it
unnoticed.

### Q7.5 — A consumer receives a message that always fails to process. What should happen?

This is a "poison message." First I need to determine whether the error is transient or permanent; if
it's permanent (e.g. malformed data), retrying is useless. The right pattern: set a retry cap with
exponential backoff and jitter; if it still fails after that, move the message to a Dead Letter Queue so
the main queue doesn't block and the rest of the messages keep processing. I then set up alerting on the
DLQ so the team is notified, can inspect and fix the message, and reprocess it if needed. Without a DLQ,
a single poison message can halt the entire pipeline.

## 7.6 Event-Driven Architecture

In an event-driven architecture, instead of calling each other directly, services publish "events" and
interested services subscribe. The main benefit: decoupling — the event producer doesn't know or care
who consumes it, and a new consumer can be added without changing the producer.

### Choreography vs. Orchestration

- **Choreography:** each service reacts to events and publishes the next event; decentralized and
  flexible, but tracing the overall flow is hard.
- **Orchestration:** a central coordinator calls each step; the flow is clear and traceable, but it's a
  single point of concentration and dependency.

### Saga for distributed transactions

In a distributed system you can't have a single ACID transaction across multiple services. The Saga
pattern breaks a business transaction into a series of local steps; if a step fails, the previous steps
are undone with a "compensating action." Example: if inventory reservation succeeds but payment fails,
the reservation is released via a compensating action.

**Note:** The price of event-driven design: eventual consistency. The system isn't immediately
consistent — it becomes consistent after events are processed. On top of that, debugging is harder
because the flow is spread across several services. A correlation id (from the Backend chapter)
becomes essential here.

### Q7.6 — What are the advantages and disadvantages of event-driven architecture?

Advantages: strong decoupling between services (the producer doesn't know about the consumer),
scalability and flexibility (adding a new consumer without changing anything else), and resilience (if
a consumer is temporarily down, the event stays in the queue). Disadvantages: more complexity, eventual
consistency (the system isn't immediately consistent), harder debugging and tracing of flow across
services, and the need to manage delivery/ordering/idempotency. For systems that need immediate strong
consistency or have simple flows, the complexity might not be justified; for large, decoupled systems
it's valuable.

## 7.7 Scaling and CAP

| Scaling type | How | Limitation |
|---|---|---|
| Vertical (scale up) | A more powerful server (more CPU/RAM) | Simple, but a hardware ceiling and a single point of failure |
| Horizontal (scale out) | More instances behind a load balancer | Requires the service to be stateless; scales nearly unbounded |

The key to horizontal scaling: the service must be stateless so any request can go to any instance.
State (session, data) must be kept in an external service (database, Redis). For a database,
sharding/partitioning splits data across nodes.

### The CAP theorem

In a distributed system, during a "network partition" (P — a break in communication between nodes), you
must choose between two of:

- **Consistency (C):** every node sees the same, up-to-date data.
- **Availability (A):** every request gets a response (possibly with stale data).

**Interview tip:** Partitions are unavoidable, so in practice it's a choice between CP and AP.
Financial systems like bunq typically lean toward Consistency (better to have an operation temporarily
unavailable than to show an incorrect balance). A social network might prefer Availability. The
decision depends on the business domain.

Consistency models exist on a spectrum: from strong consistency (always the latest) to eventual
consistency (eventually converges). Eventual is better for scale and availability but makes the logic
more complex.

### Q7.7 — How do you prepare a service for horizontal scaling?

The key is making the service stateless: it holds no important state in the instance's local memory,
so multiple instances can run behind a load balancer and any request can go to any instance. Durable
state (session, data, cache) must move to an external service (database, Redis). I offload heavy or
long-running work to a queue so workers can scale independently. For a database, which is often the
bottleneck, I use read replicas to distribute reads and sharding to distribute writes. I also need
proper idempotency and concurrency handling, since race conditions become more likely with multiple
instances.

## 7.8 Resilience Patterns

Resilience means the system degrades gracefully instead of collapsing when a component fails. A few key
patterns:

- **Timeout:** never wait forever on a dependency; always set a time ceiling so a slow dependency
  doesn't block the whole system.
- **Circuit Breaker:** if a service fails repeatedly, "trip the circuit" and stop sending requests for a
  while, giving it a chance to recover.
- **Bulkhead:** isolate resources (e.g. a separate thread pool) so a failure in one part doesn't sink
  the whole system (like a ship's watertight compartments).
- **Graceful Degradation:** when a dependency is unavailable, give a fallback response instead of an
  error (e.g. a default score instead of the model being unavailable).

### The three Circuit Breaker states

- **Closed:** normal; requests pass through and failures are counted.
- **Open:** after crossing a failure threshold, requests are rejected immediately (no call to the
  broken service).
- **Half-Open:** after a while, a few trial requests pass through; if they succeed, it returns to
  Closed, otherwise back to Open.

**Note:** Backpressure: when a system receives more work than it can handle, instead of collapsing it
should push back (e.g. rate limiting, rejecting requests with 429, or slowing consumption from a
queue). Blindly accepting all load leads to cascading failure.

### Q7.8 — What is a Circuit Breaker, and what problem does it solve?

A Circuit Breaker prevents cascading failure. The problem: if a dependent service is slow or broken and
we keep sending requests and retrying, we both exhaust our own resources (threads, connections) and put
more pressure on the already-failing service. A Circuit Breaker acts like a fuse: once failures cross a
threshold, it goes to the Open state and immediately rejects requests (fail fast) without calling the
broken service. After a while it goes Half-Open and sends a few trial requests; if they succeed, the
circuit closes and returns to normal. This gives the broken service a chance to recover and saves our
own system from wasted resources and blocking. It's usually combined with timeouts and fallbacks.

## Quick Chapter Summary (Cheat Sheet)

Review this on interview day:

- **Redis:** in-memory, atomic; cache-aside; RDB/AOF; watch for stampede (TTL+jitter) and stale data;
  cache is not the source of truth.
- **Queues:** decoupling + buffering + resilience; async is resilient but introduces eventual
  consistency.
- **Kafka vs. RabbitMQ:** Kafka = durable log + replay + high throughput; RabbitMQ = work queue +
  strong routing + transient messages.
- **Delivery:** at-least-once is common; the consumer must be idempotent; Kafka guarantees ordering
  only within a partition.
- **Retry/DLQ:** retry transient errors with backoff+jitter, not permanent ones; poison message → DLQ +
  alerting.
- **Event-driven:** decoupling; choreography vs. orchestration; Saga with compensating actions; the
  cost is eventual consistency and harder debugging.
- **Scaling:** horizontal requires statelessness; state lives in the database/Redis; sharding for
  writes, replicas for reads.
- **CAP:** during a partition, choose between C and A; financial systems typically lean CP.
- **Resilience:** timeout, circuit breaker (closed/open/half-open), bulkhead, graceful degradation,
  backpressure.